# Finetuning Large Language Models for Emoji Generation

## Introduction

In this project, we explore fine-tuning a language model for the task of emoji generation. The goal is to train a model that can take a natural language sentence as input and produce a meaningful sequence of emojis that captures the intent, tone, or context of the sentence. Emoji generation can be viewed as a lightweight sequence-to-sequence problem, where the model learns to map textual expressions to symbolic representations. By fine-tuning a pretrained language model on pairs of sentences and corresponding emoji sequences, we aim to adapt its representations to this more expressive and creative form of communication.

## Installs and Imports

In [ ]:
# Pip installs
!pip install datasets
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7

In [ ]:
# Imports
import os
import torch
from datasets import load_dataset
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)

from peft import LoraConfig, PeftModel
from trl import SFTTrainer

### Model Setup and Quantization Configuration

In [ ]:
########## Base Model + Dataset Configuration ##########

# HuggingFace model name (pretrained base model)
model_name = "NousResearch/Llama-2-7b-chat-hf"

# Dataset used for instruction fine-tuning
dataset_name = "mlabonne/guanaco-llama2-1k"

# Name for the fine-tuned model (for saving)
new_model = "llama-2-7b-miniguanaco"


########## QLoRA Parameters (Low-Rank Adaptation) ##########

# Rank of the LoRA matrices (controls size of trainable adapters)
lora_r = 64

# Scaling factor for LoRA updates
lora_alpha = 16

# Dropout applied to LoRA layers (helps prevent overfitting)
lora_dropout = 0.1


########## bitsandbytes (Quantization) Parameters ##########

# Load the base model in 4-bit precision (significantly saves memory)
use_4bit = True

# Data type used for computations (while weights are 4-bit)
bnb_4bit_compute_dtype = "float16"

# Type of 4-bit quantization:
bnb_4bit_quant_type = "nf4"    # Noraml float 4 = nf4

# Whether to use nested/double quantization (further memory savings)
use_nested_quant = False


########## TrainingArguments (HuggingFace Trainer) ##########

# Directory to save checkpoints and outputs
output_dir = "./results"

# Number of full passes through the dataset
num_train_epochs = 5

# Mixed precision settings (bf16 preferred on A100 GPUs)
fp16 = False
bf16 = False

# Batch size per GPU during training
per_device_train_batch_size = 4

# Batch size per GPU during evaluation
per_device_eval_batch_size = 4

# Number of steps to accumulate gradients before updating weights
#        Increase for super large batch sizes
gradient_accumulation_steps = 1

# Enable gradient checkpointing (reduces memory at cost of extra compute)
gradient_checkpointing = True

# Gradient clipping threshold (prevents exploding gradients)
max_grad_norm = 0.3

# Learning rate for optimizer (AdamW)
learning_rate = 2e-4

# Weight decay (regularization to prevent overfitting)
weight_decay = 0.001

# Optimizer choice (paged version is memory-efficient for large models)
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Max training steps (-1 = use epochs instead)
max_steps = -1

# Fraction of training used for learning rate warmup
warmup_ratio = 0.03

# Group sequences of similar length together (improves efficiency)
group_by_length = True

# Save checkpoints every X steps (0 = disable intermediate saving)
save_steps = 0

# Log training metrics every X steps
logging_steps = 25


########## SFT (Supervised Fine-Tuning) Parameters ##########

# Maximum sequence length (None = use model default / no truncation)
max_seq_length = None

# Whether to pack multiple short samples into one sequence (used for more efficiency)
packing = False


########## Device Configuration ##########

# Load entire model onto GPU 0
device_map = {"": 0}

In [ ]:
# Load dataset (optional preprocessing step)
#     Uncomment and modify if you want to manually process the dataset before training
# dataset = load_dataset(dataset_name, split="train")


########## bitsandbytes Configuration Setup ##########

# Convert string dtype (e.g., "float16") into actual torch dtype
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

# Create configuration for 4-bit quantization using bitsandbytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,                         # Enable 4-bit model loading
    bnb_4bit_quant_type=bnb_4bit_quant_type,       # Quantization type (nf4 or fp4)
    bnb_4bit_compute_dtype=compute_dtype,          # Computation precision
    bnb_4bit_use_double_quant=use_nested_quant,    # Nested (double) quantization
)


########## GPU Capability Check ##########

# Check if the GPU supports bfloat16 (bf16), which can speed up training
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)


########## Load Base Model ##########

# Load pretrained language model with quantization settings
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,   # Apply 4-bit quantization config
    device_map=device_map             # Load model onto specified device(s)
)

# Disable caching during training (required for gradient checkpointing)
model.config.use_cache = False

# Set tensor parallelism factor (1 = no tensor parallelism)
model.config.pretraining_tp = 1


########## Load Tokenizer ##########

# Load tokenizer corresponding to the model
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True  # Allow custom tokenizer code if needed
)

# Set padding token to EOS token (common for causal LMs like LLaMA)
tokenizer.pad_token = tokenizer.eos_token

# Ensure padding is applied to the right side of sequences
# This avoids overflow / alignment issues during fp16 training
tokenizer.padding_side = "right"

### Testing Model Before Fine-Tuning

In [ ]:
########## Testing the Original LLaMA 2 Model ##########

# Before fine-tuning, it is useful to test the base model on a few prompts.
# This gives us a baseline for how the original model responds before it is
# adapted to the emoji generation task.

# A few prompts to test general language understanding and emoji interpretation
test_prompts = [
    "What is a large language model?",
    "Can you explain this emoji 🥶",
    "Can you explain this emoji 😎",
    "What emojis would match the sentence: I am so excited for the weekend!",
    "What emojis would match the sentence: I am feeling sick today.",
    "What emojis would match the sentence: Let’s go get pizza tonight."
]

# Create a text-generation pipeline using the current model and tokenizer
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=200
)

# Run each test prompt through the model and print the generated response
for prompt in test_prompts:
    print(f"Prompt: {prompt}")

    # Format the prompt using the LLaMA 2 chat instruction style
    result = pipe(f"<s>[INST] {prompt} [/INST]")

    # Print the generated text
    print(result[0]["generated_text"])
    print("\n" + "-" * 80 + "\n")

### Memory Cleanup Utility (if needed)

In [ ]:
# # Empty VRAM
# del model
# del pipe
# # del trainer
# import gc
# gc.collect()
# gc.collect()

### Making Custom Dataset

In [ ]:
# Manualy define custom sentence and emohi labeled data
data = [('It’s a beautiful sunny day!', '☀️🌻😎'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Let’s grab a coffee.', '☕👥🤝'),
('I’m listening to music.', '🎧🎶🎵'),
('I’m listening to music.', '🎧🎶🎵'),
('Happy New Year!', '🎆🥂🎊'),
('Happy New Year!', '🎆🥂🎊'),
('Happy Birthday!', '🎉🎂🎈'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Happy Birthday!', '🎉🎂🎈'),
('Good luck on your interview!', '🍀👔💼'),
('Let’s grab a coffee.', '☕👥🤝'),
('Time for a movie night.', '🍿🎬📺'),
('I love reading books.', '📖❤️👓'),
('Time for a movie night.', '🍿🎬📺'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Merry Christmas!', '🎄🎁❄️'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Studying for exams.', '📚✍️🤓'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Ready for the football game.', '🏈📣🏟️'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Happy New Year!', '🎆🥂🎊'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Merry Christmas!', '🎄🎁❄️'),
('Going on a road trip.', '🚗🗺️🛣️'),
('I’m listening to music.', '🎧🎶🎵'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Let’s grab a coffee.', '☕👥🤝'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Studying for exams.', '📚✍️🤓'),
('Ready for the football game.', '🏈📣🏟️'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Let’s grab a coffee.', '☕👥🤝'),
('Let’s grab a coffee.', '☕👥🤝'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('Going on a road trip.', '🚗🗺️🛣️'),
('I love reading books.', '📖❤️👓'),
('Merry Christmas!', '🎄🎁❄️'),
('Studying for exams.', '📚✍️🤓'),
('Let’s grab a coffee.', '☕👥🤝'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('I’m listening to music.', '🎧🎶🎵'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Feeling under the weather.', '🤒🛌🌧️'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Happy Birthday!', '🎉🎂🎈'),
('Happy New Year!', '🎆🥂🎊'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('I’m listening to music.', '🎧🎶🎵'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('Ready for the football game.', '🏈📣🏟️'),
('Time for a movie night.', '🍿🎬📺'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('Happy New Year!', '🎆🥂🎊'),
('Let’s grab a coffee.', '☕👥🤝'),
('Happy New Year!', '🎆🥂🎊'),
('I love reading books.', '📖❤️👓'),
('Time for a movie night.', '🍿🎬📺'),
('Good luck on your interview!', '🍀👔💼'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Going on a road trip.', '🚗🗺️🛣️'),
('Happy Birthday!', '🎉🎂🎈'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Happy Birthday!', '🎉🎂🎈'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('Happy Birthday!', '🎉🎂🎈'),
('Good luck on your interview!', '🍀👔💼'),
('Ready for the football game.', '🏈📣🏟️'),
('I’m listening to music.', '🎧🎶🎵'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Let’s grab a coffee.', '☕👥🤝'),
('I love reading books.', '📖❤️👓'),
('I love reading books.', '📖❤️👓'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Studying for exams.', '📚✍️🤓'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('I’m listening to music.', '🎧🎶🎵'),
('I love reading books.', '📖❤️👓'),
('Merry Christmas!', '🎄🎁❄️'),
('Going on a road trip.', '🚗🗺️🛣️'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Time for a movie night.', '🍿🎬📺'),
('Merry Christmas!', '🎄🎁❄️'),
('I’m listening to music.', '🎧🎶🎵'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Ready for the football game.', '🏈📣🏟️'),
('I love reading books.', '📖❤️👓'),
('Let’s grab a coffee.', '☕👥🤝'),
('Ready for the football game.', '🏈📣🏟️'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Going on a road trip.', '🚗🗺️🛣️'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('Time for a movie night.', '🍿🎬📺'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Good luck on your interview!', '🍀👔💼'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('Studying for exams.', '📚✍️🤓'),
('Studying for exams.', '📚✍️🤓'),
('Going on a road trip.', '🚗🗺️🛣️'),
('Happy Birthday!', '🎉🎂🎈'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('I love reading books.', '📖❤️👓'),
('Happy Birthday!', '🎉🎂🎈'),
('Studying for exams.', '📚✍️🤓'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Happy Birthday!', '🎉🎂🎈'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('Going on a road trip.', '🚗🗺️🛣️'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Good luck on your interview!', '🍀👔💼'),
('Good luck on your interview!', '🍀👔💼'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Ready for the football game.', '🏈📣🏟️'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Happy Birthday!', '🎉🎂🎈'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('Studying for exams.', '📚✍️🤓'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('It’s a beautiful sunny day!', '☀️🌻😎'),
('Going on a road trip.', '🚗🗺️🛣️'),
('Merry Christmas!', '🎄🎁❄️'),
('I love reading books.', '📖❤️👓'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('I need a day to relax.', '🛀🧖\u200d♀️😌'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Good luck on your interview!', '🍀👔💼'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Merry Christmas!', '🎄🎁❄️'),
('Ready for the football game.', '🏈📣🏟️'),
('Happy New Year!', '🎆🥂🎊'),
('Cooking a delicious meal.', '👩\u200d🍳🥘🍴'),
('Going on a road trip.', '🚗🗺️🛣️'),
('Good luck on your interview!', '🍀👔💼'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳'),
('Merry Christmas!', '🎄🎁❄️'),
('Let’s go dancing tonight.', '💃🕺🎶'),
('Ready for the football game.', '🏈📣🏟️'),
('Good luck on your interview!', '🍀👔💼'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('I’m listening to music.', '🎧🎶🎵'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Going on a road trip.', '🚗🗺️🛣️'),
('Studying for exams.', '📚✍️🤓'),
('Time for a movie night.', '🍿🎬📺'),
('Let’s grab a coffee.', '☕👥🤝'),
('Happy New Year!', '🎆🥂🎊'),
('Happy New Year!', '🎆🥂🎊'),
('Good luck on your interview!', '🍀👔💼'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Happy New Year!', '🎆🥂🎊'),
('Time for a movie night.', '🍿🎬📺'),
('Merry Christmas!', '🎄🎁❄️'),
('I’m listening to music.', '🎧🎶🎵'),
('Time for a movie night.', '🍿🎬📺'),
('Working on my garden.', '👩\u200d🌾🌿🌷'),
('Ready for the football game.', '🏈📣🏟️'),
('Feeling under the weather.', '🤒🛌🌧️'),
('Studying for exams.', '📚✍️🤓'),
('Time for a movie night.', '🍿🎬📺'),
('Enjoy your vacation!', '✈️🏖️🍹'),
('Have a great workout!', '💪🏋️\u200d♂️🏃\u200d♀️'),
('I love reading books.', '📖❤️👓'),
('Merry Christmas!', '🎄🎁❄️'),
('Congratulations on your graduation!', '👨\u200d🎓🎓🥳')]

In [ ]:
# Convert to Datasets object
emojis_dataset = Dataset.from_dict({"text": [item[0] for item in data], "emojis": [item[1] for item in data]})

# Check to see if conversion worked
print(emojis_dataset)

### Finetune the LLM

In [ ]:
########## LoRA Configuration ##########
# Define LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,     # Scaling factor for LoRA updates
    lora_dropout=lora_dropout, # Dropout applied to LoRA layers
    r=lora_r,                  # Rank of the low-rank matrices (controls size of adapters)
    bias="none",               # Do not adapt bias terms
    task_type="CAUSAL_LM",     # Task type: causal language modeling
)

########## Training Arguments ##########
# Define HuggingFace training arguments (controls training behavior)
training_arguments = TrainingArguments(
    output_dir=output_dir,                         # Directory to save checkpoints and logs
    num_train_epochs=num_train_epochs,             # Number of full passes through dataset
    per_device_train_batch_size=per_device_train_batch_size,  # Batch size per GPU
    gradient_accumulation_steps=gradient_accumulation_steps,  # Simulate larger batch sizes
    optim=optim,                                   # Optimizer (paged AdamW for efficiency)
    save_steps=save_steps,                         # Checkpoint saving frequency
    logging_steps=logging_steps,                   # Logging frequency
    learning_rate=learning_rate,                   # Learning rate
    weight_decay=weight_decay,                     # Regularization
    fp16=fp16,                                     # Enable fp16 training
    bf16=bf16,                                     # Enable bf16 training (if supported)
    max_grad_norm=max_grad_norm,                   # Gradient clipping
    max_steps=max_steps,                           # Max steps (-1 uses epochs instead)
    warmup_ratio=warmup_ratio,                     # LR warmup fraction
    group_by_length=group_by_length,               # Batch similar-length sequences together
    lr_scheduler_type=lr_scheduler_type,           # Learning rate scheduler
    report_to="tensorboard"                        # Log metrics to TensorBoard
)


########## Trainer Setup (SFT) ##########
# Initialize the Supervised Fine-Tuning trainer
#    handles dataset loading, batching, training loop
trainer = SFTTrainer(
    model=model,                     # Base model to fine-tune
    train_dataset=emojis_dataset,    # Custom emoji dataset
    peft_config=peft_config,         # Apply LoRA adapters
    dataset_text_field="text",       # Field in dataset containing training text
    max_seq_length=max_seq_length,   # Max sequence length
    tokenizer=tokenizer,             # Tokenizer for preprocessing
    args=training_arguments,         # Training configuration
    packing=packing,                 # Whether to pack multiple samples together
)

# Start fine-tuning process
trainer.train()

# Save the trained model (including LoRA adapters)
trainer.model.save_pretrained(new_model)

In [ ]:
########## TensorBoard Monitoring (Optional) ##########

# Load the TensorBoard extension in Jupyter/Colab
#    Allows you to visualize training logs directly in the notebook
# %load_ext tensorboard

# Launch TensorBoard and point it to the training logs directory
#    "results/runs" is where HuggingFace stores logs by default when using report_to="tensorboard"
# %tensorboard --logdir results/runs

### Manual Testing of the Model


In [ ]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)


# Run text generation pipeline with our next model
#prompt = "Let’s go dancing tonight."
#pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
#result = pipe(f"<s>[INST] {prompt} [/INST]")
#print(result[0]['generated_text'])

#### Looped Testing The Model

In [ ]:
##### Known Prompt Evaluation, 1 Training Epoch #####

# Choose known prompts, 5 prompts chosen from the dataset in (prompt, answer) format
prompts_and_answers = [('Happy Birthday!', '🎉🎂🎈'), ('Let’s grab a coffee.', '☕👥🤝'), ('I’m listening to music.', '🎧🎶🎵'), ('Studying for exams.', '📚✍️🤓'), ('I love reading books.', '📖❤️👓')]

# Loop over prompts, evaluate each one
for prompt_answer_tuple in prompts_and_answers:
  prompt = prompt_answer_tuple[0]
  answer = prompt_answer_tuple[1]
  complete_prompt = f"Please describe the following prompt using only emojis: {prompt}"
  pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
  result = pipe(f"<s>[INST] {complete_prompt} [/INST]")
  print(result[0]['generated_text'])
  print(f'Answer in the dataset: {answer}')
  print()


### Testing on Novel Prompts

In [ ]:
# Define novel prompts
novel_prompts = ["I like eating carrots.", "Want to play some video games?", "Let’s watch a movie.", "Time to decorate the Christmas tree.", "Let’s take a photo together."]   # 5 prompts chosen from the dataset

# Loop over prompts, evaluate each one
for prompt in novel_prompts:
  complete_prompt = f"Please describe the following prompt using only emojis: {prompt}"
  pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
  result = pipe(f"<s>[INST] {complete_prompt} [/INST]")
  print(result[0]['generated_text'])
  print()

### Testing on Edge Case Prompts

Note: Need to add the phrase "Please describe the following prompt using only emojis:" to the beginning of each prompt in order to ensure only emogi generation

In [ ]:
# Define edge case prompts
edge_prompts = ["This sandwhich is gross but good.", "He disliked how much he liked the movie.", "Hot and cold at the same time.", "I am right next to the far away building.", "She love to watch her favorite songs."]

# Loop over prompts, evaluate each one
for prompt in edge_prompts:
  complete_prompt = f"Please describe the following prompt using only emojis: {prompt}"
  pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
  result = pipe(f"<s>[INST] {complete_prompt} [/INST]")
  print(result[0]['generated_text'])
  print()


### Evaluating a Longer Training Period (and potentially overfit model)


Train for 5 epochs over custom dataset, look for performance differences for model fine tuned on one epoch over custom dataset

In [ ]:
# Choose known prompts, 5 prompts chosen from the dataset in (prompt, answer) format
prompts_and_answers = [('Happy Birthday!', '🎉🎂🎈'), ('Let’s grab a coffee.', '☕👥🤝'), ('I’m listening to music.', '🎧🎶🎵'), ('Studying for exams.', '📚✍️🤓'), ('I love reading books.', '📖❤️👓')]

# Loop over prompts, evaluate each one
for prompt_answer_tuple in prompts_and_answers:
  prompt = prompt_answer_tuple[0]
  answer = prompt_answer_tuple[1]
  complete_prompt = f"Please describe the following prompt using only emojis: {prompt}"
  pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
  result = pipe(f"<s>[INST] {complete_prompt} [/INST]")
  print(result[0]['generated_text'])
  print(f'Answer in the dataset: {answer}')
  print()

In [ ]:
# Define novel prompts
novel_prompts = ["I like eating carrots.", "Want to play some video games?", "Let’s watch a movie.", "Time to decorate the Christmas tree.", "Let’s take a photo together."]   # 5 prompts chosen from the dataset

# Loop over prompts, evaluate each one
for prompt in novel_prompts:
  complete_prompt = f"Please describe the following prompt using only emojis: {prompt}"
  pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
  result = pipe(f"<s>[INST] {complete_prompt} [/INST]")
  print(result[0]['generated_text'])
  print()

In [ ]:
# Define edge case prompts
edge_prompts = ["This sandwhich is gross but good.", "He disliked how much he liked the movie.", "Hot and cold at the same time.", "I am right next to the far away building.", "She love to watch her favorite songs."]

# Loop over prompts, evaluate each one
for prompt in edge_prompts:
  complete_prompt = f"Please describe the following prompt using only emojis: {prompt}"
  pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
  result = pipe(f"<s>[INST] {complete_prompt} [/INST]")
  print(result[0]['generated_text'])
  print()

In [ ]:
text_prompts = ["What is the capitol of Spain?", "Who is the best player in the NBA?", "Explain the difference between a set and a list in python.", "Brefily describe how to fry an egg.", "Why is the sun so hot?"]

for prompt in text_prompts:
  pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
  result = pipe(f"<s>[INST] {prompt} [/INST]")
  print(result[0]['generated_text'])
  print()

Overfitting Note: The model when trained/finetuned for 5 epochs shows worse performance on the known prompts, which is interesting, because I would have thought that extra epochs of training would lead to better results on known prompts. Training the model for 5 epochs instead of 1 actually made it respond with text and emojis (despite specirfically asking for only emojis) more frequently when compared to training for only 1 epoch. When tested on novel prompts, the new model trained for 5 epochs performed poorly, making mistakes such as displaying an avacado instead of a carrot, and using the phone emoji for the prompt lets take a photo together. When tested on edge case prompts, the model performed better than when tested on normal novel prompts. Finally, when tested on text generation, the model performed better than expected, but still commonly used emojis in its text responses, which it did not do before the fine tuning.